# 第6章 RAG 深掘り

第2章で作った素朴な RAG を出発点に、**検索精度を上げる手法 3 つ** と **評価** を体験します。

**所要時間**: 90〜120分

## 学ぶこと
1. 複数 Loader(Markdown + PDF)で実データインデックスを作る
2. 改善手法 ① **MultiQueryRetriever**(質問を言い換えて検索漏れ防止)
3. 改善手法 ② **EnsembleRetriever**(密ベクトル + BM25 のハイブリッド)
4. 改善手法 ③ **Reranker**(取得後に並び替えて上位精度を高める)
5. RAG を **LangGraph でノード化** し、LangSmith でトレース
6. LangSmith の `evaluate` で精度を **数値で比較**

## なぜ 02 章のままだと困るのか

第 2 章の構成は **「質問をそのまま埋め込み → 上位 k 件を取って LLM に渡す」** という最小限のもので、以下の弱点があります:

| 問題 | 具体例 |
|---|---|
| **語彙ミスマッチ** | 「テレワーク」と書かれた文書を「リモートワーク」で検索すると見つかりにくい |
| **検索漏れ** | 質問の言い回し1パターンだけだと、関連チャンクが上位に来ないことがある |
| **ノイズ混入** | 上位 k に関係ないチャンクが混ざり、LLM が混乱して誤回答 |
| **複数文書源** | テキスト・PDF・Web ページなど多様な形式の文書を扱いたい |
| **精度の客観評価がない** | 「改善した」と言えても定量化できない |

本章では、これらに 1 つずつ手を打っていきます。

## 0. 準備 - 環境変数とモデル

In [ ]:
import os
from dotenv import load_dotenv

# .env を読み込む(これまでの章と同じ)。LangSmith のキーもここで読まれ、トレースが自動送信される
load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]
EMBED_MODEL_ID = os.environ["BEDROCK_EMBED_MODEL_ID"]
# Rerank モデルID。.env に無ければデフォルト値を使う(後半の Reranker セクションで利用)
RERANK_MODEL_ID = os.environ.get("BEDROCK_RERANK_MODEL_ID", "amazon.rerank-v1:0")

print("chat   :", CHAT_MODEL_ID)
print("embed  :", EMBED_MODEL_ID)
print("rerank :", RERANK_MODEL_ID)

In [ ]:
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings

# この章で使い回す2つのモデルを準備する:
#   - llm:        回答生成や、MultiQuery の「質問の言い換え」に使うチャットモデル
#   - embeddings: 文書・質問をベクトル化する埋め込みモデル(ベクトル検索の土台)
llm = ChatBedrockConverse(model=CHAT_MODEL_ID, region_name=AWS_REGION, temperature=0)
embeddings = BedrockEmbeddings(model_id=EMBED_MODEL_ID, region_name=AWS_REGION)

## 1. 複数 Loader で実データを取り込む

LangChain には文書フォーマット別の Loader が多数あります。今回は次の3種類を体験します:

| Loader | 用途 | 必要パッケージ |
|---|---|---|
| `TextLoader` | プレーンテキスト / Markdown | (本体に含む)|
| `PyPDFLoader` | PDF | `pypdf` |
| (発展)`WebBaseLoader` | Web ページ | `beautifulsoup4` |

PDF は再配布ライセンスを避けるため、Notebook 内で `reportlab` を使って簡易PDFを動的生成します。

In [ ]:
# サンプル PDF を reportlab で生成する(英語の技術文書を模した内容)
# 本物のPDFを使う場合は data/ 以下に置き、このセルはスキップしてOK
from pathlib import Path
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

pdf_path = Path("data/generated_sample.pdf")
pdf_path.parent.mkdir(parents=True, exist_ok=True)

c = canvas.Canvas(str(pdf_path), pagesize=A4)
lines = [
    "SmartLogger Pro - Architecture Overview",
    "",
    "SmartLogger Pro is composed of three main components:",
    "agents, a central indexer, and a web console.",
    "",
    "Agents are lightweight processes deployed on each",
    "target host. They tail log files and forward entries",
    "to the central indexer over TLS.",
    "",
    "The indexer stores logs in a columnar format optimized",
    "for full-text search. By default it listens on port 8443",
    "for the web console and port 9200 for agent traffic.",
    "",
    "The web console offers dashboards, alert configuration,",
    "and SAML-based single sign-on. Multi-factor authentication",
    "is enforced for all administrative accounts.",
    "",
    "High availability is supported through a primary-replica",
    "deployment. Failover is automatic, with a typical recovery",
    "time of less than 60 seconds.",
]
y = 800
for line in lines:
    c.drawString(50, y, line)
    y -= 18
c.save()
print(f"Generated: {pdf_path} ({pdf_path.stat().st_size} bytes)")

In [ ]:
# Loader はファイル形式ごとに用意されている。戻り値はどれも list[Document]:
#   - TextLoader:  プレーンテキスト / Markdown
#   - PyPDFLoader: PDF(1ページ = 1 Document になる)
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 複数 Loader の結果(list[Document])を + で連結すれば、形式の違う文書を1つにまとめられる
all_docs = []
all_docs += TextLoader("data/company_handbook.md", encoding="utf-8").load()
all_docs += TextLoader("data/product_faq.md", encoding="utf-8").load()
all_docs += PyPDFLoader("data/generated_sample.pdf").load()

print(f"元文書数(Loader直後): {len(all_docs)}")
for d in all_docs:
    # Document は .page_content(本文)と .metadata(付随情報)を持つ
    # metadata['source'] にはロード元のファイルパスが自動で入る
    print(f"  - {d.metadata.get('source', '?')}  ({len(d.page_content)}文字)")

In [ ]:
from langchain_chroma import Chroma

# 文書を検索しやすいサイズの「チャンク」に分割(第2章と同じ要領)
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)
print(f"分割後のチャンク数: {len(chunks)}")

# チャンクをベクトル化して Chroma に保存する。
# 第2章の .chroma と混ざらないよう、別ディレクトリ .chroma_v2 に保存する。
# (この vectorstore は 07章のマルチエージェントでも再利用します)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./.chroma_v2",
)
print("件数:", len(vectorstore.get()["ids"]))

> ⚠️ このセルを2回以上実行すると重複登録されます。試しに何度も実行する場合は事前に `rm -rf 06_rag_advanced/.chroma_v2` してください。

## 2. ベースライン: 素朴な top-k 検索

改善前のベースラインを作って、後で比較できるようにします。02 章と同じ単純な構成です。

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 動作確認用ヘルパー: 検索結果を見やすく表示する
def show_hits(label, hits):
    print(f"=== {label} ({len(hits)}件) ===")
    for i, d in enumerate(hits):
        source = d.metadata.get("source", "?")
        head = d.page_content.replace("\n", " ")[:100]
        print(f"  [{i}] ({source}) {head}...")

# ややひねった質問:文書には「リモートワーク」とあるが、ここでは別表現「テレワーク」で問う
question = "テレワークの可否と通信費補助について教えて"
hits = base_retriever.invoke(question)
show_hits("ベースライン", hits)

## 3. 手法① MultiQueryRetriever - 質問を言い換えて検索漏れを防ぐ

**仕組み**: LLM に「同じ意図の質問を別の言い回しで N 通り生成させる」 → 各クエリで類似検索 → 結果をマージ。
「テレワーク」だけだと拾えなくても、LLM が裏で「リモートワーク」「在宅勤務」などにも言い換えてくれます。

**コスト**: LLM を 1 回多く呼ぶ(言い換え生成)ので、応答時間とトークン消費が増えます。

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

# MultiQueryRetriever: 1つの質問を LLM に複数パターンへ言い換えさせ、
# それぞれで検索して結果をマージする Retriever。
# 語彙ミスマッチ(例: テレワーク ↔ リモートワーク ↔ 在宅勤務)に強くなる。
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,  # 各言い換えクエリの検索に使う下地の retriever
    llm=llm,                   # 言い換え文を生成する LLM
)

# MultiQuery が内部で生成した言い換えクエリを見たいので、ログレベルを INFO に上げる
# (実行すると "Generated queries: [...]" が出力に現れる)
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# ベースラインと同じ質問。言い換えのおかげで取りこぼしが減るはず
hits = mq_retriever.invoke(question)
show_hits("MultiQuery", hits)

上のログに「Generated queries: ...」と出ているはずです。LLM が複数の言い換えを生成して検索した結果がマージされています。

## 4. 手法② EnsembleRetriever - 密ベクトル + キーワード(BM25)のハイブリッド

**仕組み**: 異なる原理の Retriever を複数並列で動かし、スコアを重み付き合算して上位 N 件を返す。

| 手法 | 得意 | 苦手 |
|---|---|---|
| 密ベクトル(Chroma) | 意味的に似た文 | 固有名詞や型番のような完全一致 |
| BM25(キーワード) | 単語一致 | 言い換え |

両方を組み合わせると、お互いの弱点を補えます。

**注**: BM25Retriever は in-memory のシンプル実装で、`.from_documents()` で全チャンクを渡してインデックスを作ります。

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# BM25Retriever: キーワードの一致度でスコアリングする古典的な検索(埋め込み不要)。
# 型番・固有名詞のような「完全一致してほしい語」に強い。
# from_documents で全チャンクをその場で索引化する。
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3  # 上位何件返すか

# EnsembleRetriever: 複数の Retriever を並列実行し、スコアを重み付き合算して統合する。
#   - base_retriever (密ベクトル): 意味の近さに強い
#   - bm25_retriever (キーワード): 完全一致に強い
# 両者を混ぜることで、お互いの弱点を補う「ハイブリッド検索」になる。
ensemble_retriever = EnsembleRetriever(
    retrievers=[base_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # 各 retriever の重み(合計1.0)。チューニングポイント
)

# キーワード一致が効きそうな質問("ポート8443" という具体的な語)で比較してみる
kw_question = "ポート8443が使えないときの対処は?"
show_hits("ベースライン (dense only)", base_retriever.invoke(kw_question))
show_hits("Ensemble (dense + BM25)", ensemble_retriever.invoke(kw_question))

## 5. 手法③ Reranker - 取得した候補を並び替える

**仕組み**: 
1. まず広めに取得(例: top-20)
2. それを **Reranker モデル**(クエリと文書のペアを直接スコアリングする専用モデル)で並び替え
3. 上位 N 件(例: top-3)を最終結果として返す

埋め込みは「文書単体のベクトル化」しかしないため、クエリと文書の関連性を本当に見ているわけではありません。
Reranker はクエリと各文書を同時に評価するため、より精緻な順位付けができます。

**Bedrock Rerank**: Amazon が提供する Reranker。`amazon.rerank-v1:0` を本章のデフォルトに。

> リージョンによっては未提供のため、このセクションが動かない場合は次のセクションに進んでください。

In [ ]:
# 広めに取得する retriever (top-20)
wide_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

try:
    from langchain_aws import BedrockRerank
    from langchain.retrievers import ContextualCompressionRetriever

    # Reranker を作る(top-20 → top-3 に絞る)
    reranker = BedrockRerank(
        model_arn=RERANK_MODEL_ID,
        region_name=AWS_REGION,
        top_n=3,
    )

    # ContextualCompressionRetriever: ベース Retriever の結果を後段で「圧縮(=並び替えや要約)」する仕組み
    # ここでは reranker を圧縮器として使うことで、検索→再ランキングのパイプラインになる
    rerank_retriever = ContextualCompressionRetriever(
        base_compressor=reranker,
        base_retriever=wide_retriever,
    )

    show_hits("Reranker (top-20 → top-3)", rerank_retriever.invoke(question))
except Exception as e:
    print("⚠️ Bedrock Rerank が利用できませんでした。リージョンでサポートされていない可能性があります。")
    print(f"  詳細: {type(e).__name__}: {e}")
    print("  このセクションはスキップして次に進んで構いません。")
    rerank_retriever = None

## 6. RAG を LangGraph でノード化する

ここまでで「検索」(retriever)だけを差し替えて比較してきました。
本格的な RAG にするには **検索結果を LLM に渡して回答を生成** するチェーンが必要です。

第 2 章では LCEL で書きましたが、本章では **05 章で習った LangGraph** で書き直します。これにより:
- 各ステップ(retrieve / generate)が LangSmith で個別ノードとして見える
- 将来「ステップ追加」「条件分岐」がしやすくなる

### グラフ構造

```
  START → retrieve → generate → END
```

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate

# === 自分で State を定義する ===
# 05章では組み込みの MessagesState を使ったが、State は TypedDict で自由に定義できる。
# ここでは RAG に必要な3つのフィールドを持つ State を定義する。
# 各ノードはこの State の一部を読み、更新分(dict)を返す。
class RagState(TypedDict):
    question: str   # ユーザーの質問(入力)
    context: str    # retriever が取ってきた参考情報(retrieve ノードが埋める)
    answer: str     # LLM が生成した最終回答(generate ノードが埋める)

# retriever を引数で受け取る形にして、後で別の retriever でも同じ構造のグラフを作れるようにする
def make_rag_graph(retriever):
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "あなたは社内ドキュメントに基づいて回答するアシスタントです。\n"
         "以下の参考情報のみを根拠に、簡潔に日本語で回答してください。\n"
         "参考情報に答えがない場合は『資料には記載がありません』と答えてください。\n\n"
         "### 参考情報\n{context}"),
        ("human", "{question}"),
    ])

    # ノード1: 検索。state の question を読んで、context を埋めて返す
    def retrieve_node(state: RagState):
        docs = retriever.invoke(state["question"])
        ctx = "\n\n".join(d.page_content for d in docs)  # Document のリストを1つの文字列に
        return {"context": ctx}  # State の context フィールドだけ更新

    # ノード2: 生成。question と context を読んで、answer を埋めて返す
    def generate_node(state: RagState):
        messages = rag_prompt.format_messages(
            context=state["context"],
            question=state["question"],
        )
        response = llm.invoke(messages)
        return {"answer": response.content}

    # グラフ構築: START → retrieve → generate → END の一直線(分岐なし)
    builder = StateGraph(RagState)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("generate", generate_node)
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "generate")
    builder.add_edge("generate", END)
    return builder.compile()

# ベースラインの retriever で RAG グラフを作って実行
rag_base = make_rag_graph(base_retriever)
result = rag_base.invoke({"question": "テレワーク時の通信費補助はいくら?"})
print("質問:", result["question"])
print("回答:", result["answer"])

In [ ]:
# 同じグラフを Ensemble retriever で動かす
rag_ensemble = make_rag_graph(ensemble_retriever)
result = rag_ensemble.invoke({"question": "ポート8443で起動エラー、対処方法は?"})
print("質問:", result["question"])
print("回答:", result["answer"])

LangSmith ダッシュボードを見ると、各実行で `retrieve` ノードと `generate` ノードが独立したスパンとして記録されているはずです。
「検索結果が悪かったのか、それとも生成が悪かったのか」を区別してデバッグできるのが LangGraph 化の大きなメリットです。

## 7. LangSmith で精度を評価する

「改善した気がする」を「数値で確認できる」に進化させます。

**評価の流れ**:
1. **データセット作成**: 質問と期待される答えのペアを LangSmith に登録
2. **target 関数定義**: 入力 → 出力 を返す関数(=ここでは RAG グラフ)
3. **評価器(evaluator)定義**: 期待回答と実回答を比較してスコアを出す関数
4. **`evaluate(...)`** を呼ぶと、各サンプルが target に投入され、評価器でスコアリングされる
5. ダッシュボードでスコアを横並び比較

> `LANGSMITH_API_KEY` が `.env` に設定されている必要があります。

In [ ]:
from langsmith import Client
import uuid

client = Client()

# データセット名は毎回ユニークにする(同名で既存があると衝突するため)
dataset_name = f"rag-handson-{uuid.uuid4().hex[:8]}"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="ハンズオン用RAG評価データセット",
)

# 質問と期待される回答(キーワード)のペア
examples = [
    {"question": "テレワーク時の通信費補助はいくら?",          "expected_keywords": ["3000", "月額"]},
    {"question": "年次有給休暇の最大日数は?",                  "expected_keywords": ["20", "日"]},
    {"question": "SmartLogger Pro のスタンダード版の料金は?",  "expected_keywords": ["60", "50ノード"]},
    {"question": "パスワード変更の頻度は?",                    "expected_keywords": ["90日"]},
    {"question": "今日の東京の天気は?",                        "expected_keywords": ["記載がありません"]},
]

client.create_examples(
    inputs=[{"question": e["question"]} for e in examples],
    outputs=[{"expected_keywords": e["expected_keywords"]} for e in examples],
    dataset_id=dataset.id,
)
print(f"データセット作成: {dataset_name} ({len(examples)} 件)")

In [ ]:
from langsmith.evaluation import evaluate

# target: RAG グラフを実行する関数。inputs はデータセットの入力(dict)
def target_base(inputs: dict) -> dict:
    out = rag_base.invoke({"question": inputs["question"]})
    return {"answer": out["answer"]}

def target_ensemble(inputs: dict) -> dict:
    out = rag_ensemble.invoke({"question": inputs["question"]})
    return {"answer": out["answer"]}

# 評価器: 期待キーワードが回答に含まれていれば 1.0、そうでなければ 0.0
# シンプルだがハンズオン用には十分。本格的には LLM-as-judge も検討する
def contains_expected(outputs: dict, reference_outputs: dict) -> dict:
    answer = outputs.get("answer", "").lower()
    expected = reference_outputs.get("expected_keywords", [])
    hit = all(k.lower() in answer for k in expected)
    return {"key": "keyword_recall", "score": 1.0 if hit else 0.0}

# ベースラインを評価
result_base = evaluate(
    target_base,
    data=dataset_name,
    evaluators=[contains_expected],
    experiment_prefix="baseline",
)
print("baseline 評価完了")

In [ ]:
# Ensemble 版で同じデータセットを評価
result_ensemble = evaluate(
    target_ensemble,
    data=dataset_name,
    evaluators=[contains_expected],
    experiment_prefix="ensemble",
)
print("ensemble 評価完了")
print("\nLangSmith のダッシュボードを開いて、Datasets タブから上記データセットの Experiments を比較してください。")

## まとめ

- 複数 Loader(Markdown / PDF)を `list[Document]` として統合し、ひとつの VectorStore に保存できる
- **MultiQueryRetriever** で語彙ミスマッチに、**EnsembleRetriever** でキーワード一致の弱さに、**Reranker** で上位精度に手を打てる
- RAG を `StateGraph` でノード化すると LangSmith のスパンが分離され、デバッグしやすい
- `langsmith.evaluation.evaluate` で **客観的なスコア比較** ができる

### 発展トピック(本章スコープ外)

- **Bedrock Knowledge Bases**: マネージドな RAG。VectorStore / Embedding を自前で持たず AWS に任せる
- **Parent Document Retriever**: 検索は小さなチャンクで、LLM への提示は親文書全体で
- **HyDE**: 質問から仮想の回答を先に生成してから検索する手法
- **RAGAS**: より高度な評価指標(faithfulness、answer_relevancy 等)
- **Web 検索 RAG**: Tavily / Brave Search 等の検索API + LLM 統合

次は [07 マルチエージェント](../07_multi_agent/) で、本章で作った Retriever を **Researcher エージェント** が使う形で連携させます。